<div dir="rtl">

# مشروع الترم ١: شات بوت بسيط

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelrahman-abozarifa04/ai-for-kids-course/blob/main/notebooks/teens/term-1/03-chatbot-project.ipynb)

هنبني شات بوت **بالقواعد (rule-based)**: بيدوّر على كلمات مفتاحية في الرسالة ويرد. ده نفس الأسلوب اللي كانت بتستخدمه برامج زي **ELIZA** سنة ١٩٦٦.

في آخر النوتبوك هنشوف **حدود** الأسلوب ده، وليه الشات بوت الحديثة بتتعلم من البيانات بدل القواعد.

</div>

In [ ]:
import random
import re
from datetime import datetime

<div dir="rtl">

## ١. نجهز الرسالة

الناس بتكتب بطرق مختلفة: `Hello!!`, `hello`, `HELLO`. هنوحّدهم.

</div>

In [ ]:
def normalize(text: str) -> list[str]:
    """Lowercase, remove punctuation, split into words."""
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return text.split()

print(normalize("Hello!! How ARE you?"))

<div dir="rtl">

## ٢. القواعد: كلمات مفتاحية ← ردود

كل "نية" (intent) ليها كلمات بتدل عليها وكذا رد. الردود المتعددة بتخلي البوت ما يبقاش ممل.

</div>

In [ ]:
INTENTS = {
    "greeting": {
        "keywords": {"hi", "hello", "hey", "salam", "ahlan"},
        "responses": ["Hello! 👋", "Hi there!", "Ahlan! How can I help?"],
    },
    "how_are_you": {
        "keywords": {"how", "doing"},
        "responses": ["I'm just code, but I'm running great! And you?"],
    },
    "name": {
        "keywords": {"name", "who"},
        "responses": ["I'm Botty, a rule-based chatbot built in Term 1."],
    },
    "ai": {
        "keywords": {"ai", "intelligence", "machine", "learning"},
        "responses": ["AI is software that learns from data to make decisions.",
                      "I'm not real AI, I only follow rules. Real AI learns from examples!"],
    },
    "joke": {
        "keywords": {"joke", "funny", "laugh"},
        "responses": ["Why did the computer go to the doctor? It had a virus! 🦠",
                      "I told my computer a joke... it didn't get it. No sense of humor.exe"],
    },
    "goodbye": {
        "keywords": {"bye", "goodbye", "salam", "later"},
        "responses": ["Bye! 👋", "See you later!"],
    },
}

<div dir="rtl">

## ٣. البوت يختار النية

بنعدّ الكلمات المشتركة بين الرسالة وكل نية، وناخد النية اللي ليها أعلى عدد.

</div>

In [ ]:
def detect_intent(message: str) -> str | None:
    words = set(normalize(message))
    best, best_score = None, 0
    for intent, data in INTENTS.items():
        score = len(words & data["keywords"])
        if score > best_score:
            best, best_score = intent, score
    return best


def reply(message: str, rng: random.Random = random.Random(0)) -> str:
    words = set(normalize(message))
    if "time" in words:                       # قاعدة "ديناميكية"
        return f"It's {datetime.now():%H:%M}."
    intent = detect_intent(message)
    if intent is None:
        return "Sorry, I don't understand. Try asking about AI or a joke!"
    return rng.choice(INTENTS[intent]["responses"])

<div dir="rtl">

## ٤. نجرّب

</div>

In [ ]:
for msg in ["Hello!", "What is your name?", "Tell me a joke", "What is machine learning?",
            "What time is it?", "Can you cook koshari?", "bye"]:
    print(f"You: {msg}\nBot: {reply(msg)}\n")

<div dir="rtl">

## ٥. شات حقيقي

شغّل الخلية دي في **Colab** واكتب رسايلك. اكتب `bye` علشان تخرج.
(الخلية مقفولة بـ `INTERACTIVE = False` علشان النوتبوك يشتغل في الاختبارات الأوتوماتيك. غيّرها لـ `True`.)

</div>

In [ ]:
INTERACTIVE = False

if INTERACTIVE:
    while True:
        user = input("You: ")
        print("Bot:", reply(user))
        if detect_intent(user) == "goodbye":
            break

<div dir="rtl">

## 🏋️ تمارين المشروع

1. **ضيف نيتين جديدين** (مثلاً: `weather`, `football`).
2. **ذاكرة:** خلي البوت يسأل عن اسمك ويفتكره ("My name is Sara" ← "Nice to meet you, Sara!").
3. **عربي:** ضيف كلمات عربي أو فرانكو (`ezayak`, `3amel eh`).

<details><summary>حل تمرين ٢ (الذاكرة)</summary>

```python
memory = {}

def reply_with_memory(message):
    match = re.search(r"my name is (\w+)", message.lower())
    if match:
        memory["name"] = match.group(1).title()
        return f"Nice to meet you, {memory['name']}!"
    if "my name" in message.lower() and "name" in memory:
        return f"Your name is {memory['name']}."
    return reply(message)

print(reply_with_memory("My name is Sara"))
print(reply_with_memory("What is my name?"))
```
</details>

</div>

In [ ]:
# اكتب الحل هنا

<div dir="rtl">

## 🤔 حدود البوت ده (نقاش)

جرّب الرسايل دي وشوف البوت هيعمل إيه:

</div>

In [ ]:
for msg in ["I don't want a joke", "Who is the best football player?", "hii", "salam"]:
    print(f"You: {msg}\nBot: {reply(msg)}\n")

<div dir="rtl">

- **"I don't want a joke"**: البوت شاف كلمة joke وقال نكتة! ما بيفهمش النفي.
- **"hii"**: الكلمة مش في القاموس، فما فهمهاش.
- **"salam"**: موجودة في نيتين (greeting و goodbye)، فالبوت مش عارف يختار.

**الخلاصة:** القواعد بتتكسر بسرعة لأن اللغة البشرية فيها احتمالات لا نهائية. علشان كده الشات بوت الحديثة **بتتعلم من ملايين الأمثلة** بدل ما حد يكتبلها القواعد. ده اللي هنبدأ فيه في **الترم ٢: تعلّم الآلة**.

## ✅ معايير التقييم
| المعيار | الدرجة |
|---|---|
| البوت شغال ويرد على ٥ نوايا على الأقل | ٤٠٪ |
| ضاف ميزة جديدة (ذاكرة، عربي، وقت) | ٣٠٪ |
| عرض ٣ حدود للبوت وشرحها | ٢٠٪ |
| الكود منظم وفيه تعليقات | ١٠٪ |

</div>